# 05 - Calculating Quantum Gradients

This notebook explains how gradients move from classical machine learning to quantum machine learning.

The core question is:

$$
\text{How do we compute } \frac{\partial C(\theta)}{\partial \theta}
\text{ when } C(\theta) \text{ is produced by a quantum circuit?}
$$

We will build the intuition step by step:

1. classical gradients and gradient descent,
2. a one-qubit quantum expectation value,
3. finite differences and why they are imperfect,
4. the parameter-shift rule,
5. multi-parameter variational circuits,
6. shot noise and practical gradient estimates.

The examples use only NumPy, Matplotlib, and PennyLane from the existing `uv` environment.


## 1. Classical Gradient Intuition

In classical optimization, a gradient tells us which direction increases a function fastest.

For a scalar parameter $\theta$ and loss $L(\theta)$:

$$
\frac{dL}{d\theta}
=
\lim_{\epsilon \to 0}
\frac{L(\theta+\epsilon)-L(\theta)}{\epsilon}.
$$

Gradient descent uses the negative gradient:

$$
\theta
\leftarrow
\theta
-
\eta
\frac{dL}{d\theta},
$$

where $\eta$ is the learning rate.

If the gradient is positive, decreasing $\theta$ reduces the loss. If the gradient is negative, increasing $\theta$ reduces the loss.


In [ ]:
# Pseudo-code:
#   1. define a simple one-parameter loss
#   2. calculate its analytic gradient
#   3. run gradient descent
#   4. plot the loss curve and optimization path

# Classical one-dimensional gradient descent.
# Setup: imports and initial values for this cell.
import numpy as np
import matplotlib.pyplot as plt

def loss(theta):
    return (theta - 1.2) ** 2 + 0.25 * np.sin(4.0 * theta)

def gradient(theta):
    return 2.0 * (theta - 1.2) + np.cos(4.0 * theta)

theta = -1.7
learning_rate = 0.08
path = []

for step in range(45):
    path.append((theta, loss(theta)))
    theta = theta - learning_rate * gradient(theta)

path = np.array(path)
theta_grid = np.linspace(-2.2, 2.4, 500)

plt.figure(figsize=(8, 4.5))
plt.plot(theta_grid, loss(theta_grid), color="tab:blue", linewidth=2, label="loss L(theta)")
plt.plot(path[:, 0], path[:, 1], color="tab:red", marker="o", markersize=4, label="gradient descent path")
plt.scatter(path[0, 0], path[0, 1], color="black", s=70, label="start")
plt.scatter(path[-1, 0], path[-1, 1], color="tab:green", s=70, label="finish")
plt.title("Classical gradient descent on a one-parameter loss")
plt.xlabel("theta")
plt.ylabel("loss")
plt.grid(alpha=0.25)
plt.legend()
plt.tight_layout()
plt.show()

print("final theta:", round(float(path[-1, 0]), 3))
print("final loss:", round(float(path[-1, 1]), 3))


In [ ]:
# Pseudo-code:
#   1. repeat classical gradient descent
#   2. animate the parameter moving downhill
#   3. display the animation inside the notebook

# Animation: gradient descent as motion on a loss curve.
# Setup: imports and initial values for this cell.
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from IPython.display import HTML

def loss(theta):
    return (theta - 1.2) ** 2 + 0.25 * np.sin(4.0 * theta)

def gradient(theta):
    return 2.0 * (theta - 1.2) + np.cos(4.0 * theta)

theta = -1.7
learning_rate = 0.08
path = []
for step in range(45):
    path.append((theta, loss(theta)))
    theta = theta - learning_rate * gradient(theta)
path = np.array(path)

theta_grid = np.linspace(-2.2, 2.4, 500)
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(theta_grid, loss(theta_grid), color="tab:blue", linewidth=2)
point, = ax.plot([], [], "o", color="tab:red", markersize=9)
trail, = ax.plot([], [], color="tab:red", linewidth=1.5, alpha=0.7)
ax.set_title("Animation: following the negative gradient")
ax.set_xlabel("theta")
ax.set_ylabel("loss")
ax.grid(alpha=0.25)

def update(frame):
    point.set_data([path[frame, 0]], [path[frame, 1]])
    trail.set_data(path[:frame + 1, 0], path[:frame + 1, 1])
    return point, trail

animation = FuncAnimation(fig, update, frames=len(path), interval=120, blit=True)
plt.close(fig)
HTML(animation.to_jshtml())


## 2. A Quantum Model Is Also a Function

A variational quantum circuit produces a number after measurement. That number is a function of circuit parameters.

For one qubit:

$$
|\psi(\theta)\rangle = R_Y(\theta)|0\rangle.
$$

Measure the expectation value of $Z$:

$$
C(\theta)
=
\langle \psi(\theta)|Z|\psi(\theta)\rangle.
$$

Since

$$
R_Y(\theta)|0\rangle
=
\cos(\theta/2)|0\rangle
+
\sin(\theta/2)|1\rangle,
$$

the probabilities are

$$
P(0)=\cos^2(\theta/2),
\qquad
P(1)=\sin^2(\theta/2).
$$

Therefore,

$$
C(\theta)=P(0)-P(1)=\cos(\theta).
$$

The gradient is

$$
\frac{dC}{d\theta}=-\sin(\theta).
$$


In [ ]:
# Pseudo-code:
#   1. sweep a one-qubit rotation angle
#   2. compute expectation value and gradient
#   3. visualize state path on the Bloch X-Z circle
#   4. plot expectation and derivative

# One-qubit expectation value and gradient.
# Setup: imports and initial values for this cell.
import numpy as np
import matplotlib.pyplot as plt

theta = np.linspace(-np.pi, np.pi, 400)
expectation = np.cos(theta)
derivative = -np.sin(theta)
bloch_x = np.sin(theta)
bloch_z = np.cos(theta)

fig, axes = plt.subplots(1, 2, figsize=(11, 4.4))

axes[0].plot(bloch_x, bloch_z, color="tab:purple", linewidth=2)
axes[0].scatter([0], [1], color="black", label="theta = 0")
axes[0].set_title("RY(theta)|0> on the Bloch X-Z circle")
axes[0].set_xlabel("<X>")
axes[0].set_ylabel("<Z>")
axes[0].set_aspect("equal", adjustable="box")
axes[0].grid(alpha=0.25)
axes[0].legend()

axes[1].plot(theta, expectation, color="tab:blue", linewidth=2, label="C(theta) = cos(theta)")
axes[1].plot(theta, derivative, color="tab:red", linewidth=2, label="dC/dtheta = -sin(theta)")
axes[1].axhline(0, color="black", linewidth=1)
axes[1].set_title("Quantum circuit output is a differentiable function")
axes[1].set_xlabel("theta")
axes[1].legend()
axes[1].grid(alpha=0.25)

plt.tight_layout()
plt.show()


In [ ]:
# Pseudo-code:
#   1. animate a one-qubit state moving on the Bloch circle
#   2. show the corresponding expectation value
#   3. connect geometry to the measured scalar output

# Animation: a parameter changes both the state and the measured expectation.
# Setup: imports and initial values for this cell.
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from IPython.display import HTML

theta_values = np.linspace(0, 2 * np.pi, 80)
fig, axes = plt.subplots(1, 2, figsize=(10, 4))

circle_theta = np.linspace(0, 2 * np.pi, 300)
axes[0].plot(np.sin(circle_theta), np.cos(circle_theta), color="lightgray")
state_point, = axes[0].plot([], [], "o", color="tab:purple", markersize=10)
axes[0].set_xlim(-1.1, 1.1)
axes[0].set_ylim(-1.1, 1.1)
axes[0].set_aspect("equal", adjustable="box")
axes[0].set_title("Bloch X-Z projection")
axes[0].set_xlabel("<X>")
axes[0].set_ylabel("<Z>")
axes[0].grid(alpha=0.25)

axes[1].plot(theta_values, np.cos(theta_values), color="tab:blue", linewidth=2)
expectation_point, = axes[1].plot([], [], "o", color="tab:red", markersize=9)
axes[1].set_xlim(0, 2 * np.pi)
axes[1].set_ylim(-1.1, 1.1)
axes[1].set_title("Measured expectation C(theta)")
axes[1].set_xlabel("theta")
axes[1].set_ylabel("<Z>")
axes[1].grid(alpha=0.25)

def update(frame):
    theta = theta_values[frame]
    state_point.set_data([np.sin(theta)], [np.cos(theta)])
    expectation_point.set_data([theta], [np.cos(theta)])
    return state_point, expectation_point

animation = FuncAnimation(fig, update, frames=len(theta_values), interval=80, blit=True)
plt.close(fig)
HTML(animation.to_jshtml())


## 3. Finite Differences

The most direct numerical idea is finite differences:

$$
\frac{dC}{d\theta}
\approx
\frac{C(\theta+\epsilon)-C(\theta-\epsilon)}{2\epsilon}.
$$

This uses two circuit evaluations and works for any black-box function.

But there are two issues:

- if $\epsilon$ is too large, the approximation is biased;
- if $\epsilon$ is too small, numerical precision and shot noise become problematic.

Quantum circuits often have a better option: the **parameter-shift rule**.


## 4. The Parameter-Shift Rule

For many quantum gates of the form

$$
U(\theta)
=
e^{-i\theta G},
$$

where the generator has two distinct eigenvalues, the derivative of an expectation value can be computed exactly from shifted circuit evaluations.

For standard rotation gates such as $R_X$, $R_Y$, and $R_Z$:

$$
\frac{\partial C(\theta)}{\partial \theta}
=
\frac{1}{2}
\left[
C\left(\theta + \frac{\pi}{2}\right)
-
C\left(\theta - \frac{\pi}{2}\right)
\right].
$$

This is not a small-step approximation. For these gates, it is exact.

For $C(\theta)=\cos(\theta)$:

$$
\frac{1}{2}
\left[
\cos\left(\theta+\frac{\pi}{2}\right)
-
\cos\left(\theta-\frac{\pi}{2}\right)
\right]
=
-\sin(\theta).
$$


In [ ]:
# Pseudo-code:
#   1. choose one angle theta
#   2. evaluate C(theta), C(theta + pi/2), and C(theta - pi/2)
#   3. compute the parameter-shift gradient
#   4. compare with finite differences and analytic gradient

# Parameter-shift rule for C(theta) = cos(theta).
# Setup: imports and initial values for this cell.
import numpy as np
import matplotlib.pyplot as plt

def circuit_output(theta):
    return np.cos(theta)

theta0 = 0.85
epsilon = 0.15

analytic_gradient = -np.sin(theta0)
finite_difference = (circuit_output(theta0 + epsilon) - circuit_output(theta0 - epsilon)) / (2 * epsilon)
parameter_shift = 0.5 * (
    circuit_output(theta0 + np.pi / 2)
    -
    circuit_output(theta0 - np.pi / 2)
)

theta_grid = np.linspace(-0.3, 2.1, 400)
plt.figure(figsize=(8, 4.6))
plt.plot(theta_grid, circuit_output(theta_grid), color="tab:blue", linewidth=2, label="C(theta)")
plt.scatter([theta0], [circuit_output(theta0)], color="black", s=70, label="theta")
plt.scatter(
    [theta0 + np.pi / 2, theta0 - np.pi / 2],
    [circuit_output(theta0 + np.pi / 2), circuit_output(theta0 - np.pi / 2)],
    color="tab:red",
    s=70,
    label="parameter-shift evaluations",
)
plt.scatter(
    [theta0 + epsilon, theta0 - epsilon],
    [circuit_output(theta0 + epsilon), circuit_output(theta0 - epsilon)],
    color="tab:green",
    s=70,
    label="finite-difference evaluations",
)
plt.title("Two ways to estimate a derivative")
plt.xlabel("theta")
plt.ylabel("C(theta)")
plt.grid(alpha=0.25)
plt.legend()
plt.tight_layout()
plt.show()

print("analytic gradient:", round(float(analytic_gradient), 6))
print("finite-difference estimate:", round(float(finite_difference), 6))
print("parameter-shift estimate:", round(float(parameter_shift), 6))


In [ ]:
# Pseudo-code:
#   1. implement the one-qubit circuit in PennyLane
#   2. evaluate shifted circuits manually
#   3. compare manual parameter-shift with PennyLane differentiation

# PennyLane parameter-shift gradient on a one-qubit circuit.
# Setup: imports and initial values for this cell.
import numpy as np
import pennylane as qml
from pennylane import numpy as pnp

dev = qml.device("default.qubit", wires=1)

@qml.qnode(dev)
def expectation(theta):
    qml.RY(theta, wires=0)
    return qml.expval(qml.PauliZ(0))

theta0 = pnp.array(0.85, requires_grad=True)
manual_shift = 0.5 * (expectation(theta0 + np.pi / 2) - expectation(theta0 - np.pi / 2))

gradient_function = qml.grad(expectation)
pennylane_gradient = gradient_function(theta0)

print(qml.draw(expectation)(theta0))
print("C(theta):", round(float(expectation(theta0)), 6))
print("manual parameter-shift:", round(float(manual_shift), 6))
print("PennyLane qml.grad:", round(float(pennylane_gradient), 6))


## 5. Multi-Parameter Variational Circuits

In a variational quantum model, we usually have many parameters:

$$
C(\theta_1,\theta_2,\ldots,\theta_p).
$$

The gradient is a vector:

$$
\nabla_\theta C
=
\left[
\frac{\partial C}{\partial \theta_1},
\frac{\partial C}{\partial \theta_2},
\ldots,
\frac{\partial C}{\partial \theta_p}
\right].
$$

The parameter-shift rule is applied one parameter at a time:

$$
\frac{\partial C}{\partial \theta_j}
=
\frac{1}{2}
\left[
C(\theta_1,\ldots,\theta_j+\pi/2,\ldots)
-
C(\theta_1,\ldots,\theta_j-\pi/2,\ldots)
\right].
$$

If there are $p$ shifted parameters, the basic rule requires $2p$ circuit evaluations for one full gradient vector.


In [ ]:
# Pseudo-code:
#   1. define a two-parameter quantum circuit
#   2. evaluate the cost landscape on a grid
#   3. compute a two-component gradient by parameter shift
#   4. draw the gradient as an arrow on the landscape

# Two-parameter quantum landscape and gradient vector.
# Setup: imports and initial values for this cell.
import numpy as np
import matplotlib.pyplot as plt
import pennylane as qml

dev = qml.device("default.qubit", wires=2)

@qml.qnode(dev)
def cost(theta):
    qml.RY(theta[0], wires=0)
    qml.RX(theta[1], wires=1)
    qml.CNOT(wires=[0, 1])
    return qml.expval(qml.PauliZ(1))

def parameter_shift_gradient(theta):
    gradient = np.zeros_like(theta)
    for index in range(len(theta)):
        plus = theta.copy()
        minus = theta.copy()
        plus[index] += np.pi / 2
        minus[index] -= np.pi / 2
        gradient[index] = 0.5 * (cost(plus) - cost(minus))
    return gradient

theta0 = np.array([0.8, -0.7])
grad = parameter_shift_gradient(theta0)

grid_0 = np.linspace(-np.pi, np.pi, 120)
grid_1 = np.linspace(-np.pi, np.pi, 120)
T0, T1 = np.meshgrid(grid_0, grid_1)
Z = np.zeros_like(T0)
for row in range(T0.shape[0]):
    for col in range(T0.shape[1]):
        Z[row, col] = cost(np.array([T0[row, col], T1[row, col]]))

plt.figure(figsize=(7, 5.6))
background = plt.contourf(T0, T1, Z, levels=30, cmap="RdYlBu_r")
plt.colorbar(background, label="C(theta)")
plt.scatter([theta0[0]], [theta0[1]], color="black", s=80, label="current parameters")
plt.arrow(
    theta0[0],
    theta0[1],
    -0.8 * grad[0],
    -0.8 * grad[1],
    color="yellow",
    width=0.035,
    length_includes_head=True,
    label="negative gradient",
)
plt.title("Two-parameter quantum cost landscape")
plt.xlabel("theta[0]")
plt.ylabel("theta[1]")
plt.legend()
plt.tight_layout()
plt.show()

print("theta:", np.round(theta0, 3))
print("cost:", round(float(cost(theta0)), 6))
print("parameter-shift gradient:", np.round(grad, 6))
print("circuit evaluations for this gradient:", 2 * len(theta0))


## 6. Training with Quantum Gradients

A variational quantum classifier or regressor trains the same way as a classical model:

1. run the model,
2. compute a loss,
3. compute gradients,
4. update parameters.

For a one-qubit example, suppose the target value is $y=-0.6$ and the model is

$$
f(\theta)=\langle Z\rangle_{R_Y(\theta)|0\rangle}.
$$

Use squared error:

$$
L(\theta)
=
\left(f(\theta)-y\right)^2.
$$

By the chain rule:

$$
\frac{dL}{d\theta}
=
2(f(\theta)-y)
\frac{df}{d\theta}.
$$

The circuit derivative $df/d\theta$ can be computed by parameter-shift.


In [ ]:
# Pseudo-code:
#   1. define a one-qubit model and target value
#   2. calculate df/dtheta with parameter-shift
#   3. apply chain rule to get dL/dtheta
#   4. train with gradient descent
#   5. plot loss and parameter path

# Training a one-parameter quantum model with parameter-shift.
# Setup: imports and initial values for this cell.
import numpy as np
import matplotlib.pyplot as plt
import pennylane as qml

dev = qml.device("default.qubit", wires=1)

@qml.qnode(dev)
def model(theta):
    qml.RY(theta, wires=0)
    return qml.expval(qml.PauliZ(0))

target = -0.6
theta = 0.2
learning_rate = 0.25
history = []

for step in range(35):
    prediction = model(theta)
    loss = (prediction - target) ** 2

    shifted_derivative = 0.5 * (model(theta + np.pi / 2) - model(theta - np.pi / 2))
    loss_gradient = 2.0 * (prediction - target) * shifted_derivative

    history.append((theta, prediction, loss, loss_gradient))
    theta = theta - learning_rate * loss_gradient

history = np.array(history)
theta_grid = np.linspace(-0.2, 3.4, 400)
loss_grid = (np.cos(theta_grid) - target) ** 2

fig, axes = plt.subplots(1, 2, figsize=(11, 4.4))

axes[0].plot(theta_grid, loss_grid, color="tab:blue", linewidth=2)
axes[0].plot(history[:, 0], history[:, 2], color="tab:red", marker="o", markersize=4)
axes[0].set_title("Loss minimized with parameter-shift gradients")
axes[0].set_xlabel("theta")
axes[0].set_ylabel("loss")
axes[0].grid(alpha=0.25)

axes[1].plot(history[:, 1], color="tab:purple", linewidth=2, label="prediction")
axes[1].axhline(target, color="black", linestyle="--", label="target")
axes[1].set_title("Model output approaches the target")
axes[1].set_xlabel("step")
axes[1].set_ylabel("<Z>")
axes[1].grid(alpha=0.25)
axes[1].legend()

plt.tight_layout()
plt.show()

print("final theta:", round(float(history[-1, 0]), 4))
print("final prediction:", round(float(history[-1, 1]), 4))
print("final loss:", round(float(history[-1, 2]), 6))


## 7. Shot Noise and Gradient Uncertainty

On real hardware, expectation values are estimated from a finite number of shots.

If a circuit returns $+1$ or $-1$ measurement outcomes, then

$$
\langle Z\rangle
\approx
\frac{N_{+1}-N_{-1}}{N_{\mathrm{shots}}}.
$$

The parameter-shift estimate becomes noisy because both shifted expectation values are noisy:

$$
\frac{\partial C}{\partial \theta}
\approx
\frac{1}{2}
\left[
\hat C_{+}
-
\hat C_{-}
\right].
$$

More shots reduce variance, but they increase runtime.


In [ ]:
# Pseudo-code:
#   1. simulate finite-shot estimates of shifted expectations
#   2. repeat the gradient estimate many times
#   3. compare low-shot and high-shot uncertainty
#   4. plot histograms of gradient estimates

# Shot noise in parameter-shift gradients.
# Setup: imports and initial values for this cell.
import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(123)
theta0 = 0.85
true_gradient = -np.sin(theta0)

def sample_z_expectation(theta, shots):
    p_zero = np.cos(theta / 2) ** 2
    zero_count = rng.binomial(shots, p_zero)
    one_count = shots - zero_count
    return (zero_count - one_count) / shots

shot_settings = [50, 500, 5000]
estimates_by_shots = {}

for shots in shot_settings:
    estimates = []
    for repeat in range(400):
        plus = sample_z_expectation(theta0 + np.pi / 2, shots)
        minus = sample_z_expectation(theta0 - np.pi / 2, shots)
        estimates.append(0.5 * (plus - minus))
    estimates_by_shots[shots] = np.array(estimates)

plt.figure(figsize=(8, 4.6))
for shots, estimates in estimates_by_shots.items():
    plt.hist(estimates, bins=30, alpha=0.45, density=True, label=f"{shots} shots")
plt.axvline(true_gradient, color="black", linewidth=2, label="true gradient")
plt.title("Parameter-shift gradient estimates under shot noise")
plt.xlabel("estimated gradient")
plt.ylabel("density")
plt.grid(alpha=0.25)
plt.legend()
plt.tight_layout()
plt.show()

for shots, estimates in estimates_by_shots.items():
    print(f"{shots:5d} shots: mean={estimates.mean(): .4f}, std={estimates.std(): .4f}")


## Practical Summary

The movement from classical to quantum gradients is conceptually smooth:

| Setting | Model output | Gradient idea |
|---|---|---|
| Classical model | ordinary function $f_\theta(x)$ | backpropagation or analytic derivatives |
| Quantum model on simulator | expectation value $C(\theta)$ | automatic differentiation or parameter-shift |
| Quantum model on hardware | sampled estimate $\hat C(\theta)$ | parameter-shift with finite-shot estimates |

Key points:

- A quantum circuit plus measurement defines an ordinary numerical function.
- Finite differences are general but approximate.
- Parameter-shift is exact for many common quantum rotation gates.
- A full gradient usually needs shifted evaluations for each trainable parameter.
- On hardware, shot noise makes gradient estimates random.
- More shots reduce gradient noise but increase execution cost.

In QML, the optimizer is usually classical. The quantum device supplies function values and shifted function values.
